In [1]:
# Setup — load all dependencies
import sys
import os

# Fix path — go up one level from notebooks/ to credit-committee/
sys.path.insert(0, os.path.abspath('..'))

import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
load_dotenv(os.path.join('..', '.env'))

# Load completed modules
from src.data.financials import get_financial_ratios
from src.data.edgar import get_filing
from src.data.transcripts import get_transcript
from src.data.resolver import resolve_ticker
from src.rag.indexer import build_filing_index, query_filing
from src.schemas.models import (
    ResearchAnalystOutput,
    ManagementTone,
    RiskFactor
)

# Initialize LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.1
)

print("All imports successful")

All imports successful


In [2]:
# Load AAPL data for testing
# We already have AAPL filing downloaded and indexed from notebook 1

ticker = "AAPL"

# Step 1: Resolve ticker
print("Resolving ticker...")
resolved = resolve_ticker(ticker)
print(f"Company: {resolved['legal_name']}")
print(f"CIK: {resolved['cik']}")
print(f"Status: {resolved['status']}")

Resolving ticker...
Company: Apple Inc.
CIK: 0000320193
Status: {'success': True}


In [4]:
# Step 2: Load financial data and filing
print("Loading financial data...")
ratio_data = get_financial_ratios(ticker)
print(f"Ratios status: {ratio_data['status']}")

print("\nLoading filing...")
filing_data = get_filing(ticker, resolved['cik'])
print(f"Filing status: {filing_data['status']}")

print("\nLoading transcript...")
transcript_data = get_transcript(ticker)
print(f"Transcript status: {transcript_data['status']}")

print("\nBuilding filing index...")
index_data = build_filing_index(ticker, filing_data)
print(f"Index status: {index_data['status']}")

Loading financial data...
Ratios status: {'success': True}

Loading filing...
Filing status: {'download': 'success', 'file_found': True, 'total_chars': 3490033, 'mda_found': True, 'risk_factors_found': True, 'success': True}

Loading transcript...
Transcript status: {'transcript_url': 'https://stockanalysis.com/stocks/aapl/transcripts/548666-q2-2026/', 'char_count': 51669, 'success': True}

Building filing index...
Loading existing index for AAPL...
Index status: {'index_source': 'loaded_existing', 'success': True}


In [5]:
# Step 3: Run the 6 filing queries and collect results
FILING_QUERIES = [
    "What is the company's primary business model and main revenue sources?",
    "What does management say about liquidity, cash position, and ability to meet near-term obligations?",
    "What debt facilities, credit agreements, and covenants does the company have?",
    "Is there any going concern language from the auditor or management?",
    "What are the top risk factors disclosed by management?",
    "Has management changed language around any key risks compared to prior periods?"
]

retriever = index_data["retriever"]
query_results = {}

for i, query in enumerate(FILING_QUERIES):
    result = query_filing(retriever, query)
    query_results[f"Q{i+1}"] = result
    print(f"\nQ{i+1}: {query}")
    print(f"Result preview: {result[:200]}")
    print("---")


Q1: What is the company's primary business model and main revenue sources?
Result preview: It may be necessary in the future to seek or renew licenses relating to various aspects of the Company s products, processes and services. While the Company has generally been able to obtain such lice
---

Q2: What does management say about liquidity, cash position, and ability to meet near-term obligations?
Result preview: | 2025 Form 10-K | 43 Term Debt The Company has outstanding Notes, which are senior unsecured obligations with interest payable in arrears. The following table provides a summary of the Company s term
---

Q3: What debt facilities, credit agreements, and covenants does the company have?
Result preview: | 2025 Form 10-K | 43 Term Debt The Company has outstanding Notes, which are senior unsecured obligations with interest payable in arrears. The following table provides a summary of the Company s term
---

Q4: Is there any going concern language from the auditor or management?


In [6]:
# Step 4: Build the Research Analyst prompt and call LLM

# Format ratio summary for context
ratios = ratio_data["ratios"]
ratio_summary = f"""
Debt/EBITDA: {ratios.get('debt_to_ebitda', 'N/A')}x
Interest Coverage: {ratios.get('interest_coverage', 'N/A')}x
Current Ratio: {ratios.get('current_ratio', 'N/A')}x
FCF to Debt: {ratios.get('fcf_to_debt', 'N/A')}x
Altman Z-Score: {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})
"""

# Build context from query results
filing_context = "\n\n".join([
    f"{k}: {v[:500]}" for k, v in query_results.items()
])

# Transcript context
transcript_context = transcript_data.get("transcript_text", "")[:3000] if transcript_data["status"]["success"] else "Transcript unavailable"

# Schema description for LLM
schema_description = """
Output a JSON object with these exact fields:
{
  "business_summary": "200 word max summary of business model and revenue sources",
  "liquidity_assessment": "management stated liquidity position and cash runway",
  "covenant_observations": "key debt covenants thresholds and headroom if disclosed",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "risk description", "source": "where in filing"},
    {"rank": 2, "description": "risk description", "source": "where in filing"},
    {"rank": 3, "description": "risk description", "source": "where in filing"},
    {"rank": 4, "description": "risk description", "source": "where in filing"},
    {"rank": 5, "description": "risk description", "source": "where in filing"}
  ],
  "management_tone": "Confident",
  "language_changes": "notable changes vs prior filing or null",
  "transcript_available": true
}

management_tone must be one of: Confident, Cautious, Defensive, Distressed
going_concern_flag must be true or false
Output ONLY the JSON object. No preamble, no markdown backticks.
"""

# Build full prompt
system_prompt = """You are a senior credit research analyst at a major bank.
Your job is to extract specific qualitative credit risk signals from SEC filings 
and earnings call transcripts. Be precise, factual, and reference specific 
sections. Output ONLY valid JSON — no preamble, no markdown backticks."""

user_prompt = f"""
Company: {resolved['legal_name']}
Sector: {resolved['sector']}
Ticker: {ticker}

FINANCIAL RATIO SUMMARY:
{ratio_summary}

FILING EXCERPTS (from SEC 10-K):
{filing_context}

EARNINGS CALL TRANSCRIPT (most recent):
{transcript_context}

{schema_description}
"""

print("Calling LLM...")
print(f"Prompt length: {len(user_prompt):,} chars")

from langchain_core.messages import SystemMessage, HumanMessage

response = llm.invoke([
    SystemMessage(content=system_prompt),
    HumanMessage(content=user_prompt)
])

print("\nLLM Response:")
print(response.content)

Calling LLM...
Prompt length: 7,382 chars

LLM Response:
{
  "business_summary": "Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, Macs, and iPads, as well as services like Apple Music and Apple TV+. The company has historically experienced higher net sales in its first quarter compared to other quarters due to business seasonality and product introductions.",
  "liquidity_assessment": "Not explicitly stated in the provided excerpts, but the company's strong financial performance and cash flow suggest a stable liquidity position.",
  "covenant_observations": "The company has outstanding senior unsecured notes with interest payable in arrears, but specific covenant thresholds and headroom are not disclosed in the provided excerpts.",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "Regulatory changes and other actions that materially adversely affect the Company's business", "source": "Q6: Regulatory cha

In [7]:
# Step 5: Parse and validate with Pydantic + retry logic

def parse_with_retry(llm_response: str, max_retries: int = 2) -> ResearchAnalystOutput:
    """
    Attempts to parse LLM response as ResearchAnalystOutput.
    Retries up to max_retries times if validation fails.
    """
    
    last_error = None
    
    for attempt in range(max_retries + 1):
        try:
            # Clean response — remove markdown backticks if present
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean = clean.strip()
            
            # Fix common LLM issue — "null" string instead of None
            clean = clean.replace('"null"', 'null')
            
            # Parse JSON
            data = json.loads(clean)
            
            # Validate with Pydantic
            output = ResearchAnalystOutput(**data)
            print(f"Validation PASSED on attempt {attempt + 1}")
            return output
            
        except json.JSONDecodeError as e:
            last_error = f"JSON parse error: {e}"
            print(f"Attempt {attempt + 1} failed — {last_error}")
            
        except Exception as e:
            last_error = f"Pydantic validation error: {e}"
            print(f"Attempt {attempt + 1} failed — {last_error}")
            
            if attempt < max_retries:
                # Retry with error feedback
                print(f"Retrying with error feedback...")
                retry_prompt = f"""Your previous output failed validation with this error:
{last_error}

Please fix your output and return ONLY valid JSON matching the schema.
Previous output was:
{llm_response}

{schema_description}"""
                
                response = llm.invoke([
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=retry_prompt)
                ])
                llm_response = response.content
                print(f"New LLM response: {llm_response[:200]}")
    
    raise RuntimeError(
        f"Research Analyst failed after {max_retries + 1} attempts. "
        f"Last error: {last_error}"
    )


# Test parsing
output = parse_with_retry(response.content)

print("\nParsed Output:")
print(f"Business Summary: {output.business_summary[:100]}...")
print(f"Going Concern: {output.going_concern_flag}")
print(f"Management Tone: {output.management_tone}")
print(f"Transcript Available: {output.transcript_available}")
print(f"Risk Factors:")
for rf in output.risk_factors:
    print(f"  {rf.rank}. {rf.description[:80]}")

Validation PASSED on attempt 1

Parsed Output:
Business Summary: Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, M...
Going Concern: False
Management Tone: ManagementTone.CONFIDENT
Transcript Available: True
Risk Factors:
  1. Regulatory changes and other actions that materially adversely affect the Compan
  2. Uncertain tax positions
  3. Difficulty in obtaining licenses for various aspects of the Company's products, 
  4. Macroeconomic conditions, tariffs, and other measures
  5. Legal and regulatory proceedings


In [8]:
# Step 6: Complete run_research_analyst function

def run_research_analyst(state: dict) -> dict:
    """
    Runs the Research Analyst agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with research_analyst_output populated
    """
    
    ticker      = state["ticker"]
    legal_name  = state["legal_name"]
    sector      = state["sector"]
    
    # --- Build filing index ---
    filing_data = {
        "mda":          state.get("mda"),
        "risk_factors": state.get("risk_factors"),
        "full_text":    state.get("full_text")
    }
    
    index_data = build_filing_index(ticker, filing_data)
    
    if not index_data["status"]["success"]:
        state["errors"].append(
            f"Research Analyst: index build failed — "
            f"{index_data['status'].get('error')}"
        )
        state["current_stage"] = "research_analyst_failed"
        return state
    
    retriever = index_data["retriever"]
    
    # --- Run filing queries ---
    FILING_QUERIES = [
        "What is the company's primary business model and main revenue sources?",
        "What does management say about liquidity, cash position, and ability to meet near-term obligations?",
        "What debt facilities, credit agreements, and covenants does the company have?",
        "Is there any going concern language from the auditor or management?",
        "What are the top risk factors disclosed by management?",
        "Has management changed language around any key risks compared to prior periods?"
    ]
    
    query_results = {}
    for i, query in enumerate(FILING_QUERIES):
        try:
            query_results[f"Q{i+1}"] = query_filing(retriever, query)
        except Exception as e:
            query_results[f"Q{i+1}"] = f"Query failed: {str(e)}"
    
    # --- Build context ---
    ratios = state.get("ratio_table", {}).get("ratios", {})
    ratio_summary = f"""
Debt/EBITDA: {ratios.get('debt_to_ebitda', 'N/A')}x
Interest Coverage: {ratios.get('interest_coverage', 'N/A')}x
Current Ratio: {ratios.get('current_ratio', 'N/A')}x
FCF to Debt: {ratios.get('fcf_to_debt', 'N/A')}x
Altman Z-Score: {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})
"""
    
    filing_context = "\n\n".join([
        f"{k}: {v[:500]}" for k, v in query_results.items()
    ])
    
    transcript_text = state.get("transcript_text", "")
    transcript_available = bool(transcript_text)
    transcript_context = transcript_text[:3000] if transcript_available else "Transcript unavailable"
    
    # --- Schema description ---
    schema_description = """
Output a JSON object with these exact fields:
{
  "business_summary": "200 word max summary of business model and revenue sources",
  "liquidity_assessment": "management stated liquidity position and cash runway",
  "covenant_observations": "key debt covenants thresholds and headroom if disclosed",
  "going_concern_flag": false,
  "risk_factors": [
    {"rank": 1, "description": "risk description", "source": "where in filing"},
    {"rank": 2, "description": "risk description", "source": "where in filing"},
    {"rank": 3, "description": "risk description", "source": "where in filing"},
    {"rank": 4, "description": "risk description", "source": "where in filing"},
    {"rank": 5, "description": "risk description", "source": "where in filing"}
  ],
  "management_tone": "Confident",
  "language_changes": null,
  "transcript_available": true
}
management_tone must be one of: Confident, Cautious, Defensive, Distressed
going_concern_flag must be true or false
Output ONLY the JSON object. No preamble, no markdown backticks."""

    system_prompt = """You are a senior credit research analyst at a major bank.
Your job is to extract specific qualitative credit risk signals from SEC filings
and earnings call transcripts. Be precise, factual, and reference specific
sections. Output ONLY valid JSON — no preamble, no markdown backticks."""

    user_prompt = f"""
Company: {legal_name}
Sector: {sector}
Ticker: {ticker}

FINANCIAL RATIO SUMMARY:
{ratio_summary}

FILING EXCERPTS (from SEC 10-K):
{filing_context}

EARNINGS CALL TRANSCRIPT (most recent):
{transcript_context}

{schema_description}
"""

    # --- Call LLM with retry ---
    last_error = None
    llm_response = None
    output = None

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])
    llm_response = response.content

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean = clean.strip()
            clean = clean.replace('"null"', 'null')

            data   = json.loads(clean)
            output = ResearchAnalystOutput(**data)
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                retry_response = llm.invoke([
                    SystemMessage(content=system_prompt),
                    HumanMessage(content=f"""Your output failed validation: {last_error}
Fix and return ONLY valid JSON.
Previous output: {llm_response}
{schema_description}""")
                ])
                llm_response = retry_response.content

    if output is None:
        state["errors"].append(
            f"Research Analyst failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "research_analyst_failed"
        return state

    # --- Write to state ---
    state["research_analyst_output"] = output.model_dump()
    state["current_stage"] = "research_analyst_complete"
    
    print(f"Research Analyst complete — tone: {output.management_tone}, "
          f"going concern: {output.going_concern_flag}")
    
    return state


# --- Test with simulated state ---
test_state = {
    "ticker":         "AAPL",
    "legal_name":     resolved["legal_name"],
    "sector":         resolved["sector"],
    "mda":            filing_data["mda"],
    "risk_factors":   filing_data["risk_factors"],
    "full_text":      filing_data["full_text"],
    "transcript_text": transcript_data["transcript_text"],
    "ratio_table":    ratio_data,
    "errors":         [],
    "warnings":       []
}

updated_state = run_research_analyst(test_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"\nOutput keys: {list(updated_state['research_analyst_output'].keys())}")
print(f"\nBusiness Summary: {updated_state['research_analyst_output']['business_summary'][:200]}")

Loading existing index for AAPL...
Research Analyst complete — tone: ManagementTone.CONFIDENT, going concern: False

Stage: research_analyst_complete
Errors: []

Output keys: ['business_summary', 'liquidity_assessment', 'covenant_observations', 'going_concern_flag', 'risk_factors', 'management_tone', 'language_changes', 'transcript_available']

Business Summary: Apple Inc. is a technology company with revenue sources from the sale of products such as iPhones, Macs, and iPads, as well as services like Apple Music and Apple TV+. The company has historically exp


In [9]:
# Credit Analyst Agent
# Inputs: ratio_table from financials.py + research_analyst_output
# No ChromaDB queries needed — pure quantitative interpretation

from src.schemas.models import CreditAnalystOutput, InternalRating

CREDIT_ANALYST_SYSTEM_PROMPT = """You are a senior credit analyst at a major bank.
Your job is to interpret financial ratios and assign a preliminary internal credit rating.
Use the rating mapping guidelines provided.
Output ONLY valid JSON — no preamble, no markdown backticks."""

RATING_GUIDELINES = """
RATING MAPPING GUIDELINES:
AAA/AA : Debt/EBITDA < 1x, Interest Coverage > 15x, Altman Z > 5
A      : Debt/EBITDA 1-2x, Interest Coverage 8-15x, Altman Z > 4
BBB    : Debt/EBITDA 2-3x, Interest Coverage 4-8x, Altman Z > 3
BB     : Debt/EBITDA 3-5x, Interest Coverage 2-4x, Altman Z 2-3
B      : Debt/EBITDA 5-7x, Interest Coverage 1.5-2x, Altman Z 1.5-2
CCC    : Debt/EBITDA > 7x, Interest Coverage < 1.5x, Altman Z < 1.8
CC/C/D : Interest Coverage < 1x, negative FCF, imminent default risk

TREND ASSESSMENT:
- Compare most recent ratio vs prior periods
- Improving: ratio moving in positive direction over last 3 quarters
- Deteriorating: ratio moving in negative direction over last 3 quarters
- Stable: minimal change quarter over quarter
"""

CREDIT_ANALYST_SCHEMA = """
Output a JSON object with these exact fields:
{
  "debt_to_ebitda": 0.68,
  "interest_coverage": 33.83,
  "current_ratio": 0.89,
  "fcf_to_debt": 1.0,
  "net_debt_to_ebitda": 0.43,
  "dscr": 1.13,
  "altman_z": 29.77,
  "altman_zone": "SAFE",
  "leverage_trend": "Stable",
  "coverage_trend": "Stable",
  "cashflow_trend": "Stable",
  "quantitative_narrative": "300 word max interpretation of the quantitative picture",
  "primary_concern": "single metric most responsible for the rating",
  "preliminary_rating": "AAA",
  "preliminary_rating_rationale": "one paragraph explaining the rating decision"
}

preliminary_rating must be one of: AAA, AA, A, BBB, BB, B, CCC, CC, C, D
leverage_trend, coverage_trend, cashflow_trend must be: Improving, Stable, or Deteriorating
Output ONLY the JSON object. No preamble, no markdown backticks."""


def run_credit_analyst(state: dict) -> dict:
    """
    Runs the Credit Analyst agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with credit_analyst_output populated
    """

    ratio_table           = state.get("ratio_table", {})
    ratios                = ratio_table.get("ratios", {})
    research_output       = state.get("research_analyst_output", {})

    # --- Format ratio table ---
    ratio_context = f"""
FINANCIAL RATIOS (Latest Year: {ratio_table.get('latest_year', 'N/A')}):
| Metric              | Value                                    |
|---------------------|------------------------------------------|
| Debt/EBITDA         | {ratios.get('debt_to_ebitda', 'N/A')}x  |
| Interest Coverage   | {ratios.get('interest_coverage', 'N/A')}x|
| Current Ratio       | {ratios.get('current_ratio', 'N/A')}x   |
| FCF to Debt         | {ratios.get('fcf_to_debt', 'N/A')}x     |
| Net Debt/EBITDA     | {ratios.get('net_debt_to_ebitda', 'N/A')}x|
| DSCR                | {ratios.get('dscr', 'N/A')}x            |
| Altman Z-Score      | {ratios.get('altman_z', 'N/A')} ({ratios.get('altman_zone', 'N/A')})|
"""

    # --- Research Analyst summary for context ---
    research_context = f"""
RESEARCH ANALYST SUMMARY:
Business: {research_output.get('business_summary', 'N/A')[:300]}
Liquidity: {research_output.get('liquidity_assessment', 'N/A')[:200]}
Management Tone: {research_output.get('management_tone', 'N/A')}
Going Concern Flag: {research_output.get('going_concern_flag', False)}
"""

    user_prompt = f"""
Company: {state['legal_name']}
Sector: {state['sector']}
Ticker: {state['ticker']}

{ratio_context}

{research_context}

{RATING_GUIDELINES}

{CREDIT_ANALYST_SCHEMA}
"""

    # --- Call LLM with retry ---
    last_error   = None
    llm_response = llm.invoke([
        SystemMessage(content=CREDIT_ANALYST_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]).content
    output = None

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean  = clean.strip()
            clean  = clean.replace('"null"', 'null')
            data   = json.loads(clean)
            output = CreditAnalystOutput(**data)
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                llm_response = llm.invoke([
                    SystemMessage(content=CREDIT_ANALYST_SYSTEM_PROMPT),
                    HumanMessage(content=(
                        f"Your output failed validation: {last_error}\n"
                        f"Fix and return ONLY valid JSON.\n"
                        f"Previous output: {llm_response}\n"
                        f"{CREDIT_ANALYST_SCHEMA}"
                    ))
                ]).content

    if output is None:
        state["errors"].append(
            f"Credit Analyst failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "credit_analyst_failed"
        return state

    state["credit_analyst_output"] = output.model_dump()
    state["current_stage"]         = "credit_analyst_complete"

    print(f"Credit Analyst complete — "
          f"rating: {output.preliminary_rating}, "
          f"primary concern: {output.primary_concern}")

    return state


# --- Test ---
updated_state = run_credit_analyst(updated_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"Preliminary Rating: {updated_state['credit_analyst_output']['preliminary_rating']}")
print(f"Narrative: {updated_state['credit_analyst_output']['quantitative_narrative'][:300]}")

Credit Analyst complete — rating: InternalRating.AAA, primary concern: Current Ratio

Stage: credit_analyst_complete
Errors: []
Preliminary Rating: InternalRating.AAA
Narrative: Apple Inc.'s financial ratios indicate a strong and stable financial position. The debt-to-EBITDA ratio of 0.68x and net debt-to-EBITDA ratio of 0.43x suggest minimal leverage. The interest coverage ratio of 33.83x and DSCR of 1.13x demonstrate the company's ability to meet its financial obligations


In [10]:
# Devil's Advocate Agent
# Inputs: research_analyst_output + credit_analyst_output + ChromaDB
# Must disagree with Credit Analyst rating — counter_rating one notch below

from src.schemas.models import DevilsAdvocateOutput

RATING_ORDER = ["AAA", "AA", "A", "BBB", "BB", "B", "CCC", "CC", "C", "D"]

def enforce_counter_rating(counter_rating: str, preliminary_rating: str) -> str:
    """Ensures counter_rating is at least one notch below preliminary_rating."""
    try:
        prelim_idx  = RATING_ORDER.index(preliminary_rating)
        counter_idx = RATING_ORDER.index(counter_rating)
        # Higher index = lower rating
        if counter_idx <= prelim_idx:
            # Force one notch below
            new_idx = min(prelim_idx + 1, len(RATING_ORDER) - 1)
            return RATING_ORDER[new_idx]
        return counter_rating
    except ValueError:
        return RATING_ORDER[min(RATING_ORDER.index(preliminary_rating) + 1, 9)]


DEVIL_SYSTEM_PROMPT = """You are the Devil's Advocate in a credit committee at a major bank.
Your ONLY job is to find reasons this borrower will default.
You MUST disagree with the Credit Analyst's assessment.
Ground every argument in specific filing evidence.
Produce minimum 3, maximum 6 adversarial risk arguments.
Output ONLY valid JSON — no preamble, no markdown backticks."""

ADVERSARIAL_QUERIES = [
    "What near-term debt maturities does the company face and how does it plan to refinance?",
    "What contingent liabilities, legal proceedings, or off-balance-sheet obligations exist?",
    "What assumptions underlie management's forward guidance and how sensitive are they?",
    "What macro or industry headwinds could materially impair revenue or margins?"
]

DEVIL_SCHEMA = """
Output a JSON object with these exact fields:
{
  "adversarial_risks": [
    {
      "rank": 1,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing supporting this",
      "quantitative_impact": "estimated financial impact if risk materializes",
      "probability": "High"
    },
    {
      "rank": 2,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing",
      "quantitative_impact": "estimated impact",
      "probability": "Medium"
    },
    {
      "rank": 3,
      "risk_description": "description of risk",
      "filing_evidence": "specific text from filing",
      "quantitative_impact": "estimated impact",
      "probability": "Low"
    }
  ],
  "most_likely_default_trigger": "single most plausible path to default in 12-24 months",
  "bear_case_narrative": "200 word max bear case summary",
  "counter_rating": "AA",
  "macro_vulnerability": "how a macro downturn specifically hurts this borrower"
}
adversarial_risks must have minimum 3 items
probability must be one of: High, Medium, Low
counter_rating must be one of: AAA, AA, A, BBB, BB, B, CCC, CC, C, D
Output ONLY the JSON object. No preamble, no markdown backticks."""


def run_devil_advocate(state: dict) -> dict:
    """
    Runs the Devil's Advocate agent.
    Input: state (dict) — LangGraph pipeline state
    Output: updated state dict with devils_advocate_output populated
    """

    ticker          = state["ticker"]
    research_output = state.get("research_analyst_output", {})
    credit_output   = state.get("credit_analyst_output", {})
    preliminary_rating = credit_output.get(
        "preliminary_rating", "BBB"
    ).replace("InternalRating.", "")

    # --- Load index and run adversarial queries ---
    filing_data = {
        "mda":          state.get("mda"),
        "risk_factors": state.get("risk_factors"),
        "full_text":    state.get("full_text")
    }
    index_data = build_filing_index(ticker, filing_data)
    retriever  = index_data["retriever"]

    adversarial_context = {}
    for i, query in enumerate(ADVERSARIAL_QUERIES):
        try:
            adversarial_context[f"AQ{i+1}"] = query_filing(retriever, query)
        except Exception as e:
            adversarial_context[f"AQ{i+1}"] = f"Query failed: {str(e)}"

    filing_evidence = "\n\n".join([
        f"{k}: {v[:400]}" for k, v in adversarial_context.items()
    ])

    user_prompt = f"""
Company: {state['legal_name']}
Sector: {state['sector']}
Ticker: {ticker}

CREDIT ANALYST PRELIMINARY RATING: {preliminary_rating}
You MUST assign a counter_rating that is LOWER (worse) than {preliminary_rating}.

RESEARCH ANALYST OUTPUT:
Business: {research_output.get('business_summary', 'N/A')[:200]}
Risk Factors: {str(research_output.get('risk_factors', []))[:300]}
Management Tone: {research_output.get('management_tone', 'N/A')}
Going Concern: {research_output.get('going_concern_flag', False)}

CREDIT ANALYST OUTPUT:
Rating: {preliminary_rating}
Primary Concern: {credit_output.get('primary_concern', 'N/A')}
Narrative: {credit_output.get('quantitative_narrative', 'N/A')[:200]}

ADVERSARIAL FILING EVIDENCE:
{filing_evidence}

MACRO CONTEXT:
{state.get('macro_narrative', 'Not available')}

{DEVIL_SCHEMA}
"""

    last_error   = None
    llm_response = llm.invoke([
        SystemMessage(content=DEVIL_SYSTEM_PROMPT),
        HumanMessage(content=user_prompt)
    ]).content
    output = None

    for attempt in range(3):
        try:
            clean = llm_response.strip()
            if clean.startswith("```"):
                clean = clean.split("```")[1]
                if clean.startswith("json"):
                    clean = clean[4:]
            clean  = clean.strip()
            clean  = clean.replace('"null"', 'null')
            data   = json.loads(clean)
            output = DevilsAdvocateOutput(**data)

            # Enforce counter_rating constraint
            output.counter_rating = enforce_counter_rating(
                str(output.counter_rating).replace("InternalRating.", ""),
                preliminary_rating
            )
            break

        except Exception as e:
            last_error = str(e)
            if attempt < 2:
                llm_response = llm.invoke([
                    SystemMessage(content=DEVIL_SYSTEM_PROMPT),
                    HumanMessage(content=(
                        f"Your output failed validation: {last_error}\n"
                        f"Fix and return ONLY valid JSON.\n"
                        f"Previous output: {llm_response}\n"
                        f"{DEVIL_SCHEMA}"
                    ))
                ]).content

    if output is None:
        state["errors"].append(
            f"Devil's Advocate failed after 3 attempts: {last_error}"
        )
        state["current_stage"] = "devils_advocate_failed"
        return state

    state["devils_advocate_output"] = output.model_dump()
    state["current_stage"]          = "devils_advocate_complete"

    print(f"Devil's Advocate complete — "
          f"counter_rating: {output.counter_rating}, "
          f"default trigger: {output.most_likely_default_trigger[:80]}")

    return state


# --- Test ---
updated_state = run_devil_advocate(updated_state)

print(f"\nStage: {updated_state['current_stage']}")
print(f"Errors: {updated_state['errors']}")
print(f"Counter Rating: {updated_state['devils_advocate_output']['counter_rating']}")
print(f"Bear Case: {updated_state['devils_advocate_output']['bear_case_narrative'][:300]}")
print(f"\nAdversarial Risks:")
for risk in updated_state['devils_advocate_output']['adversarial_risks']:
    print(f"  {risk['rank']}. {risk['risk_description'][:80]}")

Loading existing index for AAPL...
Devil's Advocate complete — counter_rating: AA, default trigger: Regulatory changes leading to a decline in iPhone sales

Stage: devils_advocate_complete
Errors: []
Counter Rating: AA
Bear Case: Apple's heavy reliance on iPhone sales and regulatory changes could lead to a decline in revenue, making it difficult for the company to meet its debt obligations. Additionally, uncertain tax positions could result in increased tax liability, further straining the company's financials.

Adversarial Risks:
  1. Regulatory changes affecting business
  2. Uncertain tax positions
  3. Default on senior unsecured obligations


c:\Users\Dev Vatsayan\credit-committee\venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `enum` - serialized value may not be as expected [field_name='counter_rating', input_value='AA', input_type=str])
  return self.__pydantic_serializer__.to_python(
